# MIDASML -- SG-LASSO MIDAS with Weekly Data (Pipeline B)

**Model**: MIDASML via `midasml::cv.sglfit`
**Target**: `gdpc1`
**Variables**: Cat3 (53 + COVID = 56) + 4 weekly = 60 total
**Train**: **1967-2007** / **Val**: 2008-2016 / **Test**: 2017-2025
**HP tuning**: YES -- gamma=0.25, degree=3 via CV
**Frequency handling**:
  - Weekly: high_freq_lags=12, mixed_freq_data with weekly dates
  - Monthly/Quarterly: high_freq_lags=9, mixed_freq_data with monthly dates
**Fixes applied**:
  1. Train start 1967 (aligns with weekly data)
  2. Inf check on weekly data
  3. Weekly vars integrated via separate mixed_freq_data calls


In [1]:
options(warn = -1)
library(tidyverse)
library(midasml)
library(imputeTS)

source("../data/helpers.R")

metadata <- read_csv("../data/meta_data.csv")
data <- read_csv("../data/data_tf_monthly.csv") %>% arrange(date)
data_weekly <- read_csv("../data/data_tf_weekly.csv") %>%
    rename(date = Date) %>% arrange(date)

# Inf check on weekly data
for (col in colnames(data_weekly)) {
    if (col == "date") next
    if (sum(is.infinite(data_weekly[[col]])) > 0) {
        data_weekly[is.infinite(data_weekly[[col]]), col] <- NA
    }
}

cat3_features <- c(
  "a014re1q156nbea","acogno","ahetpix","amdmuox","andenox","awotman",
  "busloans","ce16ov","ces1021000001","ces2000000008","ces9091000001",
  "ces9092000001","clf16ov","compapff","cusr0000sas","ddurrg3m086sbea",
  "dhlcrg3q086sbea","difsrg3q086sbea","dodgrg3q086sbea","dongrg3q086sbea",
  "dspic96","expgsc1","fpix","gcec1","gpdic1","houstne","housts",
  "hwiuratio","hwiuratiox","invest","ipdcongd","liabpix","lns13023705",
  "m2sl","manemp","mortg10yrx","nonrevsl","ophpbs","outbs","outnfb",
  "permitw","realln","slcex","spcs10rsa","tlbsnncbbdix","uemp15t26",
  "uemp27ov","uemplt5","ulcbs","ulcnfb","unrate","usgovt","usserv",
  "covid_2020q2","covid_2020q3","covid_2020q4",
  "icsa_weekly","nfci_weekly","dtwexbgs","dtwexm"
)
cat3_features <- tolower(cat3_features)

freq_lookup <- metadata %>%
    dplyr::filter(series %in% cat3_features) %>%
    select(series, freq)
weekly_vars <- freq_lookup %>% dplyr::filter(freq == "w") %>% pull(series)
cat("Weekly vars:", weekly_vars, "
")

# Modified gen_midasml_dataset: weekly vars handled via weekly CSV
# with separate high_freq_lags and date alignment
gen_midasml_dataset_v2 <- function(data, data_weekly, target_variable,
    degree, low_freq_lags, high_freq_lags_monthly,
    high_freq_lags_weekly, train_start_date, train_end_date,
    weekly_vars, cat3_features_in) {

    cov_cols <- colnames(data)[!(colnames(data) %in% c(target_variable, "date"))]
    cov_cols <- cov_cols[cov_cols %in% cat3_features_in]

    # Append weekly covariates from data_weekly (not in monthly data)
    for (wcol in weekly_vars) {
        if (wcol %in% colnames(data_weekly)) {
            cov_cols <- c(cov_cols, wcol)
            # Fill weekly data (NA handled by imputeTS)
            if (sum(is.na(data_weekly[[wcol]])) > 0) {
                data_weekly[[wcol]] <- na_mean(data_weekly[[wcol]])
            }
        }
    }

    # Fill monthly/quarterly covariates
    for (covariate in cov_cols) {
        if (covariate != target_variable && !(covariate %in% weekly_vars)) {
            data[, covariate] <- na_mean(data[, covariate])
        }
    }

    mfd <- list()
    for (covariate in cov_cols) {
        if (covariate %in% weekly_vars) {
            hl <- high_freq_lags_weekly
            # Use weekly dates and values from data_weekly
            mfd[[covariate]] <- mixed_freq_data(
                data[, target_variable],
                as.Date(data[["date"]]),
                data_weekly[, covariate],
                as.Date(data_weekly[["date"]]),
                                x.lag = hl, y.lag = low_freq_lags,
                horizon = 1, train_start_date, train_end_date,
                disp.flag = FALSE)
        } else {
            hl <- high_freq_lags_monthly
            mfd[[covariate]] <- mixed_freq_data(
                data[, target_variable],
                as.Date(data[["date"]]),
                data[, covariate],
                as.Date(data[["date"]]),
                x.lag = hl, y.lag = low_freq_lags,
                horizon = 1, train_start_date, train_end_date,
                disp.flag = FALSE)
        }
    }

    y <- mfd[[1]]$est.y
    x <- mfd[[1]]$est.lag.y
    for (covariate in cov_cols) {
        hl <- if (covariate %in% weekly_vars) high_freq_lags_weekly
              else high_freq_lags_monthly
        w <- lb(degree = degree, jmax = hl)
        x <- cbind(x, mfd[[covariate]]$est.x %*% w)
    }

    # gindex: direct adaptation of original gen_midasml_dataset.
    # Groups: 1 for lagged-y block + 1 per covariate. Each group has dim(est.lag.y)[2] entries.
    # (lb(degree,jmax) returns degree+1 cols, so each block has the same width as est.lag.y)
    gindex <- c()
    for (i in 1:(length(cov_cols) + 1)) {
        gindex <- c(gindex, rep(i, dim(mfd[[1]]$est.lag.y)[2]))
    }
    return(list(y = y, x = x, gindex = gindex))
}

target_variable <- "gdpc1"
gamma <- 0.25
degree <- 3
low_freq_lags <- 4
high_freq_lags_monthly <- 9
high_freq_lags_weekly <- 12

train_start_date <- "1967-01-01"    # aligned with weekly data start
test_start_date  <- "2017-03-01"
test_end_date    <- "2025-12-01"
test_dates <- seq(as.Date(test_start_date), as.Date(test_end_date), by = "3 months")

test <- data %>%
    dplyr::filter(date >= as.Date(train_start_date), date <= as.Date(test_end_date)) %>% data.frame()

for (col in colnames(test)) {
    if (sum(is.infinite(test[, col])) > 0) test[is.infinite(test[, col]), col] <- NA
}

vintage_offsets <- c(m1 = -2, m2 = -1, m3 = 0)
pred_dict <- data.frame(date = test_dates)
for (lag_name in names(vintage_offsets)) pred_dict[, lag_name] <- NA


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Zorunlu paket yükleniyor: Matrix




Attaching package: 'Matrix'




The following objects are masked from 'package:tidyr':

    expand, pack, unpack




...


ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Rows: 3095 Columns: 8


── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl  (7): icsa_weekly, nfci_weekly, dtwexbgs, dtwexm, covid_2020q2, covid_20...
date (1): Date



ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Weekly vars: icsa_weekly nfci_weekly dtwexbgs dtwexm 


In [2]:
for (i in 1:length(test_dates)) {
    train_end <- shift_month(test_dates[i], -3)
    train <- test %>%
        dplyr::filter(date <= train_end)

    dataset <- gen_midasml_dataset_v2(train, data_weekly, target_variable,
        degree, low_freq_lags, high_freq_lags_monthly,
        high_freq_lags_weekly, train_start_date, train_end,
        weekly_vars, cat3_features)

    model <- cv.sglfit(dataset$x, dataset$y, gamma = gamma,
                       gindex = dataset$gindex, nfold = 5)

    for (lag_name in names(vintage_offsets)) {
        vintage_date <- shift_month(test_dates[i], vintage_offsets[[lag_name]])
        data_weekly_vintage <- data_weekly
        data_weekly_vintage[data_weekly_vintage$date > vintage_date, weekly_vars] <- NA
        lagged_data <- gen_vintage_data(metadata, test, test_dates[i], vintage_date)
        lagged_data <- data.frame(lagged_data)
        lagged_data[lagged_data$date == test_dates[i], target_variable] <- NA

        for (j in c(0, 3, 6, 9)) {
            idx <- nrow(lagged_data) - j
            if (idx > 0 && is.na(lagged_data[idx, target_variable])) {
                lagged_data[idx, target_variable] <- mean(
                    lagged_data[, target_variable], na.rm = TRUE)
            }
        }

        lagged_dataset <- tryCatch(
            gen_midasml_dataset_v2(lagged_data, data_weekly_vintage, target_variable,
                degree, low_freq_lags, high_freq_lags_monthly,
                high_freq_lags_weekly, train_start_date, test_dates[i],
                weekly_vars, cat3_features),
            error = function(e) NULL)

        pred <- tryCatch({
            p <- predict(model, newx = lagged_dataset$x, s = "lam.min")
            p[nrow(p)]
        }, error = function(e) NA)

        pred_dict[pred_dict$date == test_dates[i], lag_name] <- pred
    }
    if (i %% 8 == 0) print(paste(i, "/", length(test_dates)))
}

actuals <- test %>%
    dplyr::filter(date >= as.Date(test_start_date)) %>%
    dplyr::filter(substr(date, 6, 7) %in% c("03", "06", "09", "12")) %>%
    select(!!target_variable) %>% pull()


[1] "8 / 36"
[1] "16 / 36"
[1] "24 / 36"
[1] "32 / 36"


In [3]:
dir.create("../predictions", showWarnings = FALSE)
for (lag_name in names(vintage_offsets)) {
    df_out <- data.frame(date = test_dates, actual = actuals,
                         prediction = pred_dict[, lag_name])
    write.csv(df_out, paste0("../predictions/midasml_", lag_name, ".csv"), row.names = FALSE)
}

panels <- list(
    "Pre-COVID"  = c("2017-01-01", "2019-12-31"),
    "COVID"      = c("2020-04-01", "2021-12-31"),
    "Post-COVID" = c("2022-01-01", "2025-12-31"),
    "Full"       = c("2017-01-01", "2025-12-31")
)
rmse <- function(a, p) sqrt(mean((a - p)^2, na.rm = TRUE))
mae  <- function(a, p) mean(abs(a - p), na.rm = TRUE)
d <- test_dates
for (pn in names(panels)) {
    rng <- panels[[pn]]; m <- d >= rng[1] & d <= rng[2]
    cat("--- ", pn, " ---
")
    for (lag_name in names(vintage_offsets)) {
        cat("  ", lag_name, "  RMSFE=",
            format(rmse(actuals[m], pred_dict[m, lag_name]), digits=6), "  MAE=",
            format(mae(actuals[m], pred_dict[m, lag_name]), digits=6), "
")
    }
}


---  Pre-COVID  ---
   m1   RMSFE= 0.00292225   MAE= 0.0023298 
   m2   RMSFE= 0.00293008   MAE= 0.0023376 
   m3   RMSFE= 0.00292736   MAE= 0.0023353 
---  COVID  ---
   m1   RMSFE= 0.0434678   MAE= 0.0282336 
   m2   RMSFE= 0.0434213   MAE= 0.0287468 
   m3   RMSFE= 0.0434016   MAE= 0.028444 
---  Post-COVID  ---
   m1   RMSFE= 0.00453686   MAE= 0.0033148 
   m2   RMSFE= 0.00496029   MAE= 0.00374203 
   m3   RMSFE= 0.00526677   MAE= 0.00392593 
---  Full  ---
   m1   RMSFE= 0.0198034   MAE= 0.00833569 
   m2   RMSFE= 0.0198284   MAE= 0.00862732 
   m3   RMSFE= 0.0198547   MAE= 0.00864918 
